In [2]:
import pandas as pd

# one hot encoding
df = pd.DataFrame({
    'Neighbourhood': ['Riverside','Downtown','Suburbs','Riverside','Downtown'],
    'Price':         [250000,380000,210000,265000,395000]
})

df_encoded = pd.get_dummies(df, columns=['Neighbourhood'], drop_first=True)
print(df_encoded)
# Creates: Neighbourhood_Riverside, Neighbourhood_Suburbs
# Downtown is dropped (reference category)

# On Titanic — Embarked has 3 values: S, C, Q
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df  = pd.read_csv(url)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
print([c for c in df.columns if 'Embarked' in c])

    Price  Neighbourhood_Riverside  Neighbourhood_Suburbs
0  250000                     True                  False
1  380000                    False                  False
2  210000                    False                   True
3  265000                     True                  False
4  395000                    False                  False
['Embarked_Q', 'Embarked_S']


In [ ]:
# target encoding

from sklearn.model_selection import train_test_split
import category_encoders as ce

df = pd.DataFrame({
    'Neighbourhood': ['Riverside','Downtown','Suburbs','Riverside','Downtown','Suburbs'],
    'Price':         [250000,380000,210000,265000,395000,195000]
})

train, test = train_test_split(df, test_size=0.33, random_state=42)

print(train.shape)
print(test.shape)

# Step 1: calculate mean target per category from TRAIN only
target_map = train.groupby('Neighbourhood')['Price'].mean()

# Step 2: apply to both train and test
train['Neighbourhood_enc'] = train['Neighbourhood'].map(target_map)
test['Neighbourhood_enc'] = test['Neighbourhood'].map(target_map)

print(train)

encoder   = ce.TargetEncoder(cols=['Neighbourhood'], smoothing=1.0)
train_enc = encoder.fit_transform(train[['Neighbourhood']], train['Price'])
test_enc  = encoder.transform(test[['Neighbourhood']])

(4, 2)
(2, 2)
  Neighbourhood   Price  Neighbourhood_enc
5       Suburbs  195000           202500.0
2       Suburbs  210000           202500.0
4      Downtown  395000           395000.0
3     Riverside  265000           265000.0
   Neighbourhood
5  266249.999029
2  266249.999029
4  266250.000721
3  266249.999993


In [12]:
# outlier detection
import numpy as np

def detect_outliers(series: pd.Series):
    Q1, Q3 = series.quantile(0.25) , series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper= Q3 + 1.5 * IQR


    print(f"Bounds: {lower:,.0f} to {upper:,.0f}")
    print(f"Outliers: {((series < lower) | (series > upper)).sum()}")
    return lower, upper

def cap_outliers(series: pd.Series):
    lower, upper = detect_outliers(series)
    
    return series.clip(lower=lower, upper=upper)

# Log transformation — log1p handles zero values safely
prices     = pd.Series([150000, 200000, 250000, 300000, 5000000])
lower,upper = detect_outliers(prices)
print(lower,upper)
prices_log = np.log1p(prices)
print("Log:", prices_log.round(2).values)


# Reverse the log transformation after predicting
predictions_log  = np.array([12.5, 12.8, 13.1])
predictions_real = np.expm1(predictions_log)
print(np.expm1(13.1))
print("Predicted prices:", predictions_real.round(0))




Bounds: 50,000 to 450,000
Outliers: 1
50000.0 450000.0
Log: [11.92 12.21 12.43 12.61 15.42]
488941.41461545986
Predicted prices: [268336. 362216. 488941.]


In [ ]:
# interaction and polynomial features

